In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web

In [42]:
def get_stock(ticker, start_date, end_date, s_window, l_window):
    try:
        #yf.pdr_override()  # Uncomment if necessary
        df = yf.download(ticker, start=start_date, end=end_date)
        # Add additional columns
        if 'Adj Close' not in df.columns:
            df['Adj Close'] = df['Close']

        #df = web.get_data_yahoo(ticker, start=start_date, end=end_date)
        df['Return'] = df['Adj Close'].pct_change()
        df['Return'].fillna(0, inplace=True)
        df['Date'] = df.index
        df['Date'] = pd.to_datetime(df['Date'])
        df['Month'] = df['Date'].dt.month
        df['Year'] = df['Date'].dt.year
        df['Day'] = df['Date'].dt.day
        for col in ['Open', 'High', 'Low', 'Close', 'Adj Close']:
            df[col] = df[col].round(2)
        df['Weekday'] = df['Date'].dt.day_name()
        df['Week_Number'] = df['Date'].dt.strftime('%U')
        df['Year_Week'] = df['Date'].dt.strftime('%Y-%U')
        df['Short_MA'] = df['Adj Close'].rolling(window=s_window, min_periods=1).mean()
        df['Long_MA'] = df['Adj Close'].rolling(window=l_window, min_periods=1).mean()
        col_list = ['Date', 'Year', 'Month', 'Day', 'Weekday',
                    'Week_Number', 'Year_Week', 'Open',
                    'High', 'Low', 'Close', 'Volume', 'Adj Close',
                    'Return', 'Short_MA', 'Long_MA']
        num_lines = len(df)
        df = df[col_list]
        print('Read', num_lines, 'lines of data for ticker:', ticker)
        return df
    except Exception as error:
        print(error)
        return None

try:
    ticker = "XLY"
    start_date = '2019-01-01'
    end_date = '2024-12-31'
    input_dir = os.getcwd()
    output_file = os.path.join(input_dir, f'{ticker}.csv')
    df = get_stock(ticker, start_date, end_date, s_window=14, l_window=50)
    if df is not None:
        df.to_csv(output_file, index=False)
        print(f'Wrote {len(df)} lines to file: {output_file}')
except Exception as e:
    print(e)
    print('Failed to get Yahoo stock data for ticker:', ticker)


[*********************100%***********************]  1 of 1 completed

Read 1509 lines of data for ticker: XLY
Wrote 1509 lines to file: /content/XLY.csv



<ipython-input-42-24d812b9fac3>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Return'].fillna(0, inplace=True)


In [43]:
import os
import pandas as pd

ticker='XLY'
input_dir = os.getcwd()  # Use current working directory or specify a path
ticker_file = os.path.join(input_dir, ticker + '.csv')

try:
    with open(ticker_file) as f:
        lines = f.read().splitlines()
    print('opened file for ticker: ', ticker)
    """    your code for assignment 1 goes here
    """

except Exception as e:
    print(e)
    print('failed to read stock data for ticker: ', ticker)
### Part - 1: Preliminary Assignment

opened file for ticker:  XLY


In [44]:
import os
import numpy as np
import pandas as pd

# Load the modified CSV file
df = pd.read_csv("XLY.csv")
df = df.iloc[1:]
df.to_csv("XLY.csv", index=False)
# Ensure the 'Date' column is in datetime format
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
# Add additional columns
if 'Adj Close' not in df.columns:
    df['Adj Close'] = df['Close']

# Calculate daily returns and handle NaNs
df['Return'] = df['Adj Close'].pct_change().fillna(0) * 100
df['Return'] = df['Return'].round(3)

# Add columns for week number and year
df['Week_Number'] = df.index.strftime('%U')
df['Year'] = df.index.year

# Select relevant columns for grouping
df_2 = df[['Year', 'Week_Number', 'Return']]
df_2.reset_index(drop=True, inplace=True)

# Group by year and week number to calculate mean return and volatility
df_grouped = df_2.groupby(['Year', 'Week_Number'])['Return'].agg(['mean', 'std'])
df_grouped.reset_index(inplace=True)
df_grouped.rename(columns={'mean': 'mean_return', 'std': 'volatility'}, inplace=True)
df_grouped.fillna(0, inplace=True)

# Save the weekly return and volatility data to a new CSV file
output_file = "XLY_weekly_return_volatility.csv"
df_grouped.to_csv(output_file, index=False)
print("Wrote", len(df_grouped), "lines to file:", output_file)

# Save detailed data by merging original and grouped DataFrames
try:
    # Ensure the 'Week_Number' is a string for both DataFrames
    df['Week_Number'] = df['Week_Number'].astype(str)
    df_grouped['Week_Number'] = df_grouped['Week_Number'].astype(str)

    # Merge DataFrames on 'Year' and 'Week_Number'
    combined_df = df.reset_index().merge(df_grouped, on=['Year', 'Week_Number'], how='inner')
    detailed_output_file = "XLY_weekly_return_detailed.csv"
    combined_df.to_csv(detailed_output_file, index=False)
    print("\nWrote", len(combined_df), "lines to file:", detailed_output_file)

except Exception as e:
    print("Error during merge:", e)


Wrote 315 lines to file: XLY_weekly_return_volatility.csv

Wrote 1509 lines to file: XLY_weekly_return_detailed.csv
